In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, initcap, upper, to_date, 
    regexp_replace, current_timestamp, lit
)
from pyspark.sql.types import DoubleType, IntegerType
import logging

INFO:py4j.clientserver:Received command c on object id p1


In [0]:
dbutils.secrets.list("DBSecrets")

[SecretMetadata(key='adls'),
 SecretMetadata(key='databricks'),
 SecretMetadata(key='dbsid'),
 SecretMetadata(key='dbsid1'),
 SecretMetadata(key='github'),
 SecretMetadata(key='kaggle')]

In [0]:
# ============================================================================
# 1. SETUP & AUTHENTICATION
# ============================================================================
spark = SparkSession.builder \
    .appName("Healthcare_Bronze_to_Silver_Parquet") \
    .getOrCreate()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- CONFIGURATION ---
STORAGE_ACCOUNT = "adlstoragesiddharth"
STORAGE_KEY = dbutils.secrets.get("DBSecrets", "adls")
CONTAINER_BRONZE = "bronze"
CONTAINER_SILVER = "silver"

# Paths
ADLS_BRONZE_PATH = f"abfss://{CONTAINER_BRONZE}@{STORAGE_ACCOUNT}.dfs.core.windows.net/Healthcare_Dataset"
ADLS_SILVER_PATH = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/healthcare/patient_data"

# Set Authentication Context
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net", 
    STORAGE_KEY
)

In [0]:

# ============================================================================
# 2. INGESTION (BRONZE)
# ============================================================================
logger.info("Reading raw CSV data from Bronze...")

try:
    df_raw = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("quote", "\"") \
        .option("escape", "\"") \
        .load(ADLS_BRONZE_PATH)
except Exception as e:
    logger.error(f"Error reading Bronze: {str(e)}")
    raise

INFO:__main__:Reading raw CSV data from Bronze...
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


In [0]:

# ============================================================================
# 3. TRANSFORMATION & CLEANING (Keeping original column names)
# ============================================================================
logger.info("Applying cleaning logic...")

df_cleaned = (df_raw
    .withColumn("Name", initcap(trim(col("Name"))))
    .withColumn("Gender", initcap(trim(col("Gender"))))
    .withColumn("Blood Type", upper(trim(col("Blood Type"))))
    .withColumn("Medical Condition", initcap(trim(col("Medical Condition"))))
    .withColumn("Doctor", initcap(trim(col("Doctor"))))
    .withColumn("Insurance Provider", initcap(trim(col("Insurance Provider"))))
    .withColumn("Medication", initcap(trim(col("Medication"))))
    .withColumn("Test Results", initcap(trim(col("Test Results"))))
    
    # Hospital cleaning
    .withColumn("Hospital", regexp_replace(col("Hospital"), r'[",]', ""))
    .withColumn("Hospital", initcap(trim(col("Hospital"))))
    
    # Type Casting
    .withColumn("Age", col("Age").cast(IntegerType()))
    .withColumn("Billing Amount", col("Billing Amount").cast(DoubleType()))
    .withColumn("Room Number", col("Room Number").cast(IntegerType()))
    
    # Dates
    .withColumn("Date of Admission", to_date(col("Date of Admission")))
    .withColumn("Discharge Date", to_date(col("Discharge Date")))
    
    # Basic Filters
    .filter((col("Age") > 0) & (col("Age") < 120))
    .filter(col("Billing Amount") >= 0)
    .dropDuplicates()
    
    # Audit column
    .withColumn("processed_at", current_timestamp())
)


INFO:__main__:Applying cleaning logic...
INFO:py4j.clientserver:Received command c on object id p0


In [0]:

# ============================================================================
# 4. LOAD (SILVER - PARQUET FORMAT)
# ============================================================================
logger.info(f"Writing to Silver in Parquet format: {ADLS_SILVER_PATH}")

try:
    # Changed format from "delta" to "parquet"
    df_cleaned.write.format("parquet") \
        .mode("overwrite") \
        .save(ADLS_SILVER_PATH)
    
    logger.info("Pipeline Execution Successful!")
except Exception as e:
    logger.error(f"Error writing Parquet to Silver: {str(e)}")
    raise



INFO:__main__:Writing to Silver in Parquet format: abfss://silver@adlstoragesiddharth.dfs.core.windows.net/healthcare/patient_data
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Python Server ready to receive messages
INFO:py4j.clientserver:Received command c on object id p0
INFO:__main__:Pipeline Execution Successful!


In [0]:
# ============================================================================
# 5. VERIFY
# ============================================================================
final_df = spark.read.format("parquet").load(ADLS_SILVER_PATH)
final_df.select("Name", "Age", "Blood Type", "Hospital").show(5)

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


+---------------+---+----------+--------------------+
|           Name|Age|Blood Type|            Hospital|
+---------------+---+----------+--------------------+
|Brandon Collins| 77|        O+|           Lopez Plc|
|   Jeffrey Wood| 81|       AB-|         Brown-yoder|
| Michael Jordan| 30|       AB-|     Hernandez-green|
| Melinda Tanner| 38|       AB-|            Ball Llc|
|  Brian Osborne| 30|        B-|Parker Turner Hou...|
+---------------+---+----------+--------------------+
only showing top 5 rows
